# Quantum-bug RAG evaluation — v6 (deprecation KB + framework-aware retrieval)

v6 builds on v5 with two targeted improvements identified through actual sample-by-sample failure analysis of v5's wrong predictions.

## v5 results recap
| Dataset | Mode | top-1 | top-2 |
|---|---|---|---|
| bugs4q | rag | 44.4% | 75.6% |
| bugsqcp | prompt_only | 56.8% | 95.5% |
| bugsqcp | rag | 56.8% | 90.9% |

## v6 changes (both directly target making RAG > prompt-only)

**Change 1 — KB now includes deprecation/upgrade entries.** Sample analysis of v5 wrong predictions showed many Bugs4Q failures involved deprecated Qiskit APIs (`compile()`, `qasm_simulator_py`, `iden()`, old `save_statevector` semantics) that gpt-4o doesn't recognize because they were removed before its training cutoff. v5's KB only used `fixes:` sections from release notes — missing 170 `deprecations:` and 403 `upgrade:` entries that explicitly document deprecated/removed APIs. v6 includes all three sections. **This adds knowledge gpt-4o doesn't already have, which is exactly when RAG provides lift.**

**Change 2 — Framework-aware retrieval.** v5's retriever sometimes pulled `pennylane_changelog` entries when classifying Qiskit code (and vice versa) because BM25 doesn't know about frameworks. v6 detects each sample's framework via syntactic signatures (`qiskit.`, `cirq.`, `qml.`, `.qs`), then prioritizes KB entries from the matching framework while keeping cross-framework entries as fallback for diversity. **This reduces noise that was actively hurting RAG vs prompt-only on Bugs-QCP.**

## What top-2 accuracy means (since you asked)
Forced-choice classification has the model output likelihood scores 0.0-1.0 for all 5 specific classes. **Top-1 accuracy** = was the highest-scored class correct? **Top-2 accuracy** = was the right answer in the top 2 highest-scored classes? Top-2 is a standard metric in classification with intrinsically ambiguous categories (ImageNet uses top-5 always; medical NLP uses top-3 or top-5). It measures how close the model came to the right answer when categories overlap (e.g., a CNOT swap is plausibly both `incorrect_operator` and `incorrect_qubit_mapping`).

## What I expect for v6
Realistic targets: **bugs4q ~50-55%, bugsqcp ~60-65% top-1**. Top-2 should stay around 95% on bugsqcp. RAG should now beat prompt-only on both datasets. **Still not promising 70%+** but this directly addresses the failure patterns from v5's diagnostic JSONLs.

## Cell 1 — Clone source repos

In [1]:
import os, subprocess, sys
from pathlib import Path

WORK = Path("/kaggle/working")
WORK.mkdir(parents=True, exist_ok=True)
os.chdir(WORK)

REPOS = {
    "bugs4q_upstream":   "https://github.com/Z-928/Bugs4Q.git",
    "bqcp":              "https://github.com/MattePalte/Bugs-Quantum-Computing-Platforms.git",
    "qiskit":            "https://github.com/Qiskit/qiskit.git",
    "qiskit_aer":        "https://github.com/Qiskit/qiskit-aer.git",
    "qiskit_ignis":      "https://github.com/Qiskit/qiskit-ignis.git",
    "qiskit_ibm_runtime":"https://github.com/Qiskit/qiskit-ibm-runtime.git",
    "pennylane":         "https://github.com/PennyLaneAI/pennylane.git",
}
for name, url in REPOS.items():
    target = WORK / name
    if target.exists():
        print(f"  {name}: already cloned")
        continue
    print(f"  cloning {name} ...")
    r = subprocess.run(["git", "clone", "--depth", "1", url, str(target)],
                       capture_output=True, text=True)
    if r.returncode != 0:
        raise RuntimeError(f"Clone failed: {r.stderr[-300:]}")
print(f"\n✓ All {len(REPOS)} repos ready.")

  cloning bugs4q_upstream ...
  cloning bqcp ...
  cloning qiskit ...
  cloning qiskit_aer ...
  cloning qiskit_ignis ...
  cloning qiskit_ibm_runtime ...
  cloning pennylane ...

✓ All 7 repos ready.


## Cell 2 — Install deps

In [2]:
!pip install -q rank_bm25 openai pyyaml
import rank_bm25, openai, numpy, sklearn, yaml
print(f"✓ rank_bm25, openai {openai.__version__}, numpy {numpy.__version__}, sklearn {sklearn.__version__}, pyyaml {yaml.__version__}")

✓ rank_bm25, openai 2.23.0, numpy 2.0.2, sklearn 1.6.1, pyyaml 6.0.3


## Cell 3 — Schemas

**v5 change:** TAXONOMY_FORCED has only 5 classes (no `unknown`). Used for forced-choice classification.

In [3]:
import json, re, random, subprocess, time, yaml
from dataclasses import dataclass, asdict, field
from pathlib import Path
from typing import Optional
from collections import Counter, defaultdict

@dataclass
class BugSample:
    sample_id: str
    source: str
    code: str
    ground_truth: Optional[str] = None
    metadata: dict = field(default_factory=dict)

@dataclass
class BugDiagnostic:
    sample_id: str
    mode: str
    bug_likelihood: float
    taxonomy_class: str
    class_scores: dict = field(default_factory=dict)  # NEW: per-class likelihoods
    suspected_location: str = ""
    justification: str = ""
    ground_truth: Optional[str] = None
    correct: Optional[bool] = None
    retrieved_patterns: list = field(default_factory=list)

@dataclass
class BugPattern:
    pattern_id: str
    name: str
    taxonomy_class: str
    description: str
    example_code: str = ""
    fix_hint: str = ""
    source: str = ""
    tags: list = field(default_factory=list)

# 5 specific classes — no 'unknown' here. unknown is for KB tagging only.
TAXONOMY_FORCED = [
    "incorrect_operator", "incorrect_qubit_mapping", "missing_barrier",
    "wrong_initial_state", "measurement_error",
]
# For metric computation: includes 'unknown' as fallback if parser can't extract a class
TAXONOMY_ALL = TAXONOMY_FORCED + ["unknown"]

print(f"✓ Schemas + 5-class forced taxonomy ready: {TAXONOMY_FORCED}")

✓ Schemas + 5-class forced taxonomy ready: ['incorrect_operator', 'incorrect_qubit_mapping', 'missing_barrier', 'wrong_initial_state', 'measurement_error']


## Cell 4 — Bugs4Q adapter (unchanged)

In [4]:
BUGS4Q_TYPE_MAP = {
    "parameter": "incorrect_operator", "qr,qc": "incorrect_qubit_mapping",
    "empty circuit": "incorrect_operator", "qasm": "incorrect_operator",
    "output wrong": "measurement_error", "wrong circuit design": "incorrect_operator",
    "wrong command": "incorrect_operator",
    "being not familiar with the usage of measuring all bit using existing registers.": "measurement_error",
    "qiskit distinguishes operations in `gate`s": "incorrect_operator",
    "quantumcircuit.parameters` only tracks unbound parameters.": "incorrect_operator",
    "not fully understanding qasm and statevector/eval computation.": "wrong_initial_state",
    "the circuit library requires `decompose` for \"lin_comb\".": "incorrect_operator",
    "ignoring the impact of measurement": "measurement_error",
    "order during measurement": "measurement_error",
    "oversized resource consumption": "measurement_error",
    "unfamiliar with api": "incorrect_operator", "figure problem": "incorrect_operator",
    "name conflict": "incorrect_operator",
    "label convention is reversed(|011>&|110>)": "wrong_initial_state",
    "wrong operation with gate": "incorrect_operator",
    "qft operation*": "incorrect_operator", "qfe output wrong": "measurement_error",
    "ccx": "incorrect_operator", "random gates": "incorrect_operator",
    "not a dag": "incorrect_operator", "wait()": "incorrect_operator",
    "grover algrithm": "incorrect_operator", "only for simulator": "measurement_error",
    "compiler() removerd": "incorrect_operator", "obtain amplitude": "measurement_error",
    "output": "measurement_error", "threads": "incorrect_operator",
    "statevector": "wrong_initial_state", "initialization": "wrong_initial_state",
    "`transpile` required": "incorrect_operator", "transpile` required": "incorrect_operator",
    "outdated grammar": "incorrect_operator", "start state is reversed": "wrong_initial_state",
    "no output": "measurement_error", "random number error": "wrong_initial_state",
    "wrong circuit operation": "incorrect_operator", "call wrong function": "incorrect_operator",
}
_LINK_RE = re.compile(r"\[[^\]]*\]\(([^)]+)\)")
_FIXED_FN = {"fixed.py", "fix.py", "fixed_version.py", "modify.py", "mod.py"}

def _map_bugs4q_type(t):
    if not t: return None
    return BUGS4Q_TYPE_MAP.get(re.sub(r"\s+", " ", t.lower().strip()))

def _parse_bugs4q_readme(readme_path):
    cases = {}
    if not readme_path.exists(): return cases
    headers, buggy_idx, type_idx = None, None, None
    for line in readme_path.read_text(encoding="utf-8", errors="replace").splitlines():
        s = line.strip()
        if not s.startswith("|"): continue
        cells = [c.strip() for c in s.strip("|").split("|")]
        lower = [c.lower() for c in cells]
        if "buggy" in lower and "type" in lower:
            headers, buggy_idx, type_idx = cells, lower.index("buggy"), lower.index("type")
            continue
        if headers is None or all(set(c) <= {"-", ":"} for c in cells): continue
        if len(cells) > len(headers):
            trail = max(len(headers) - type_idx - 1, 0)
            end = len(cells) - trail
            cells = cells[:type_idx] + [" | ".join(cells[type_idx:end])] + cells[end:]
        if len(cells) <= max(buggy_idx, type_idx): continue
        bcell, tcell = cells[buggy_idx], cells[type_idx].strip("`").strip()
        for tgt in _LINK_RE.findall(bcell):
            cleaned = tgt.replace("\\", "/").lstrip("./").rstrip("/")
            if cleaned:
                cases[cleaned] = tcell if tcell and tcell != "---" else None
                break
    return cases

def _is_buggy_file(py):
    n = py.name.lower()
    if n in _FIXED_FN: return False
    if n in {"buggy.py", "bug_version.py"} or n.startswith("bug"): return True
    if "buggy" in {p.lower() for p in py.parts}: return True
    return False

def build_bugs4q(repo_root):
    cases = _parse_bugs4q_readme(repo_root / "README.md")
    sorted_paths = sorted(cases.keys(), key=len, reverse=True)
    def lookup_type(rel):
        rel = str(rel).replace("\\", "/").lstrip("./")
        for cp in sorted_paths:
            if rel == cp or rel.startswith(cp + "/"): return cases[cp]
        return None
    samples = []
    for py in sorted(repo_root.rglob("*.py")):
        if not _is_buggy_file(py): continue
        try: code = py.read_text(encoding="utf-8", errors="replace")
        except Exception: continue
        if not code.strip(): continue
        rel = py.relative_to(repo_root)
        raw_type = lookup_type(rel)
        label = _map_bugs4q_type(raw_type)
        samples.append(BugSample(
            sample_id=f"bugs4q_{len(samples):04d}",
            source="bugs4q", code=code, ground_truth=label,
            metadata={"path": str(rel), "bug_pattern_raw": raw_type},
        ))
    return samples

bugs4q_samples = build_bugs4q(WORK / "bugs4q_upstream")
labelled = sum(1 for s in bugs4q_samples if s.ground_truth)
print(f"✓ Bugs4Q: {len(bugs4q_samples)} samples, {labelled} labelled")
print(f"  ground_truth: {dict(Counter(s.ground_truth for s in bugs4q_samples))}")

✓ Bugs4Q: 99 samples, 45 labelled
  ground_truth: {'measurement_error': 12, 'wrong_initial_state': 5, 'incorrect_operator': 27, None: 54, 'incorrect_qubit_mapping': 1}


## Cell 5 — Bugs-QCP focused-snippet adapter (unchanged from v3)

In [5]:
import csv
BQCP_PATTERN_MAP = {
    "qubit mapping": "incorrect_qubit_mapping", "qubit index": "incorrect_qubit_mapping",
    "index": "incorrect_qubit_mapping", "operator": "incorrect_operator",
    "gate": "incorrect_operator", "wrong gate": "incorrect_operator",
    "barrier": "missing_barrier", "initial": "wrong_initial_state",
    "statevector": "wrong_initial_state", "measurement": "measurement_error",
    "measure": "measurement_error", "barrier related": "missing_barrier",
    "overlooked qubit order": "incorrect_qubit_mapping",
    "msb-lsb convention": "incorrect_qubit_mapping",
    "msb-lsb convention mismatches": "incorrect_qubit_mapping",
    "wrong identifier": "incorrect_qubit_mapping",
    "incorrect numerical computation": "incorrect_operator",
    "incorrect ir - wrong information": "incorrect_operator",
    "incorrect ir - missing information": "incorrect_operator",
    "incorrect circuit": "incorrect_operator",
    "api misuse - internal": "incorrect_operator",
    "api misuse - external": "incorrect_operator", "api misuse": "incorrect_operator",
    "typo": "incorrect_operator", "type problem": "incorrect_operator",
    "incorrect final measurement": "measurement_error",
    "incorrect randomness handling": "wrong_initial_state",
}
def _map_bqcp(raw):
    if not raw: return None
    for piece in (p.strip().lower() for p in raw.split(",")):
        m = BQCP_PATTERN_MAP.get(piece)
        if m and m != "unknown": return m
    return None

def _build_bqcp_index(fixes_root):
    index = {}
    for mp in fixes_root.rglob("metadata.json"):
        try: meta = json.loads(mp.read_text(encoding="utf-8"))
        except Exception: continue
        proj, bid = meta.get("project_name"), meta.get("id")
        if proj and bid is not None:
            index[(proj, str(bid))] = mp.parent
    return index

def extract_focused_snippet(folder: Path, context_lines: int = 10) -> Optional[str]:
    before_dir = folder / "before"
    after_dir = folder / "after"
    if not before_dir.exists() or not after_dir.exists(): return None
    try:
        d = subprocess.run(["diff", "-u", f"-U{context_lines}", "-r", str(before_dir), str(after_dir)],
                           capture_output=True, text=True, timeout=15)
    except Exception: return None
    if not d.stdout.strip(): return None
    per_file = defaultdict(list)
    current_file, current = None, []
    for line in d.stdout.splitlines():
        if line.startswith("--- "):
            if current and current_file:
                per_file[current_file].append("\n".join(current)); current = []
            full = line[4:].split("\t")[0]
            current_file = full.replace(str(before_dir) + "/", "").replace(str(before_dir), "")
        elif line.startswith("+++"): continue
        elif line.startswith("@@"):
            if current and current_file:
                per_file[current_file].append("\n".join(current)); current = []
            current.append(f"# {line}")
        elif line.startswith("+"): continue
        elif line.startswith("-"): current.append(line[1:])
        elif line.startswith(" "): current.append(line[1:])
    if current and current_file:
        per_file[current_file].append("\n".join(current))
    if not per_file: return None
    deletions_per_file = defaultdict(int)
    cur_f = None
    for line in d.stdout.splitlines():
        if line.startswith("--- "):
            full = line[4:].split("\t")[0]
            cur_f = full.replace(str(before_dir) + "/", "").replace(str(before_dir), "")
        elif line.startswith("-") and not line.startswith("---") and cur_f:
            deletions_per_file[cur_f] += 1
    best_file = max(per_file.keys(), key=lambda f: deletions_per_file.get(f, 0) or len(per_file[f]))
    hunks = [h for h in per_file[best_file] if h.strip()]
    if not hunks: return None
    return f"# === {best_file} ===\n" + "\n\n".join(hunks)

def build_bugsqcp(repo_root, *, quantum_only=True):
    csv_path = repo_root / "artifacts" / "annotation_bugs.csv"
    fixes_root = repo_root / "artifacts" / "minimal_bugfixes"
    index = _build_bqcp_index(fixes_root)
    samples = []
    with csv_path.open(encoding="utf-8") as fh:
        for row in csv.DictReader(fh):
            if row["real"] != "bug": continue
            if quantum_only and row["type"].strip().lower() != "quantum": continue
            suffix = row["repo"].split("/")[-1]
            folder = None
            for cand in (suffix, suffix.lower(), suffix.capitalize()):
                if (cand, row["id"]) in index:
                    folder = index[(cand, row["id"])]; break
            if folder is None: continue
            snippet = extract_focused_snippet(folder, context_lines=10)
            if not snippet: continue
            samples.append(BugSample(
                sample_id=f"bqcp_{row['id'].replace(',', '_'):>06}",
                source="bugsqcp", code=snippet,
                ground_truth=_map_bqcp(row["bug_pattern"]),
                metadata={"repo": row["repo"], "bug_pattern_raw": row["bug_pattern"], "type": row["type"]},
            ))
    return samples

bqcp_samples = build_bugsqcp(WORK / "bqcp", quantum_only=True)
labelled = sum(1 for s in bqcp_samples if s.ground_truth)
print(f"✓ Bugs-QCP (Quantum-only): {len(bqcp_samples)} samples, {labelled} labelled")
print(f"  ground_truth: {dict(Counter(s.ground_truth for s in bqcp_samples))}")

✓ Bugs-QCP (Quantum-only): 89 samples, 44 labelled
  ground_truth: {'incorrect_operator': 33, 'missing_barrier': 2, None: 45, 'incorrect_qubit_mapping': 9}


## Cell 6 — Build expanded validated KB (v6: includes deprecation/upgrade sections)

Three changes from v5's KB build:
1. Extract from **3 YAML sections**: `fixes`, `deprecations`, `upgrade` (was: only `fixes`).
2. PennyLane changelog: also extract `Breaking Changes` / `Deprecations` sections (was: only Bug Fixes).
3. IBM Runtime: also extract `Deprecation Notes` / `Upgrade Notes` sections.

Each entry tagged with `[fixes]` or `[deprecations]` so retrieval can stratify if needed. Net effect: KB grows from 1059 (v5) to ~1600 entries, with the new content directly addressing the deprecated-API knowledge gap.


In [6]:
TAX_KEYWORDS = {
    "measurement_error": [
        "measure", "measurement", "classical bit", "cbit", "counts",
        "qasm_simulator", "shots", "final_measurement",
        "remove_final_measurements", "clbit", "bit_string",
    ],
    "incorrect_qubit_mapping": [
        "qubit index", "qubit order", "register", "msb", "lsb",
        "cnot", "control qubit", "target qubit", "wires", "qargs", "cargs",
        "layout", "qubit mapping", "physical qubit", "virtual qubit",
        "qubit count", "swap mapper",
    ],
    "missing_barrier": [
        "barrier", "reorder", "transpiler reorder", "optimization across",
        "removed by optimization",
    ],
    "wrong_initial_state": [
        "initialize", "statevector", "state preparation", "amplitude",
        "normaliz", "initial state", "random_unitary", "rand_circuit",
        "prep_state", "save_statevector",
    ],
    "incorrect_operator": [
        "gate", "operator", "rotation", "matrix", "transpile",
        "decomposition", "iden ", "identity gate", "unitary", "parameter",
        "deprecated", "controlled gate", "conditional gate",
        "angle", "radians", "degrees", "u1", "u2", "u3", "swap gate",
        "pauli", "hadamard", "phase", "compile", "removed", "renamed",
    ],
}

def classify_text_to_taxonomy(text):
    text_l = text.lower()
    scores = {cls: sum(1 for kw in kws if kw in text_l) for cls, kws in TAX_KEYWORDS.items()}
    best_cls, best_score = max(scores.items(), key=lambda x: x[1])
    return best_cls if best_score > 0 else "unknown"

def _is_low_quality_entry(text):
    if len(text) < 80 or len(text) > 1500:
        return True
    text_l = text.lower()
    if re.search(r"renamed to", text_l) and len(text) < 200:
        return True
    if any(p in text_l for p in ["bump version", "updated dependency", "removed deprecated alias for"]):
        return True
    return False

# v6: Each KB entry tagged with framework so framework-aware retrieval can use it
SOURCE_TO_FRAMEWORK = {
    "qiskit_releasenotes": "qiskit",
    "qiskit_aer_releasenotes": "qiskit",
    "qiskit_ignis_releasenotes": "qiskit",
    "ibm_runtime_changelog": "qiskit",
    "pennylane_changelog": "pennylane",
    "lintq_rules": "qiskit",  # LintQ targets Qiskit
}

# v6: extract from MULTIPLE YAML sections (was: only "fixes")
YAML_SECTIONS = ["fixes", "deprecations", "upgrade"]

def extract_yaml_releasenote_entries(repo_root, source_label, notes_subdir="releasenotes/notes"):
    """v6: extract from fixes + deprecations + upgrade sections."""
    patterns = []
    notes_dir = repo_root / notes_subdir
    if not notes_dir.exists():
        return patterns
    framework = SOURCE_TO_FRAMEWORK.get(source_label, "other")
    for yf in notes_dir.rglob("*.yaml"):
        try:
            d = yaml.safe_load(yf.read_text(encoding="utf-8"))
        except Exception:
            continue
        if not isinstance(d, dict):
            continue
        for section_name in YAML_SECTIONS:
            entries = d.get(section_name)
            if not entries or not isinstance(entries, list):
                continue
            for i, item in enumerate(entries):
                if not isinstance(item, str):
                    continue
                text = item.strip()
                cleaned = re.sub(r":[a-z_]+:`([^`]+)`", r"\1", text)
                cleaned = re.sub(r"`([^`]+)`_", r"\1", cleaned)
                cleaned = re.sub(r"<https?://[^>]+>", "", cleaned)
                cleaned = re.sub(r"\s+", " ", cleaned).strip()
                if _is_low_quality_entry(cleaned):
                    continue
                cls = classify_text_to_taxonomy(cleaned)
                if cls == "unknown":
                    continue
                patterns.append(BugPattern(
                    pattern_id=f"{source_label}_{section_name}_{yf.stem}_{i}",
                    name=f"{source_label} {section_name}: {yf.stem[:40]}",
                    taxonomy_class=cls,
                    description=cleaned[:800],
                    source=source_label,
                    tags=[source_label, "validated", section_name, framework],
                ))
    return patterns

# PennyLane: extract Bug Fixes AND Breaking Changes/Deprecations sections
def extract_pennylane_changelog_entries(pl_root):
    patterns = []
    changelog_dir = pl_root / "doc" / "releases"
    if not changelog_dir.exists():
        return patterns
    for md in changelog_dir.glob("changelog-*.md"):
        text = md.read_text(encoding="utf-8", errors="replace")
        sections = re.split(r"<h3>([^<]+)</h3>", text)
        for i in range(1, len(sections), 2):
            heading = sections[i].lower()
            body = sections[i + 1] if i + 1 < len(sections) else ""
            if "bug" in heading or "fix" in heading:
                section_tag = "fixes"
            elif "deprecat" in heading or "breaking" in heading or "removed" in heading:
                section_tag = "deprecations"
            else:
                continue
            for j, m in enumerate(re.finditer(r"\n\* +(.+?)(?=\n\* |\n<h3>|\Z)", body, re.DOTALL)):
                item = m.group(1).strip()
                cleaned = re.sub(r"```[^`]*```", "", item, flags=re.DOTALL)
                cleaned = re.sub(r"\[\(#\d+\)\]\([^)]+\)", "", cleaned)
                cleaned = re.sub(r"\s+", " ", cleaned).strip()
                if _is_low_quality_entry(cleaned):
                    continue
                cls = classify_text_to_taxonomy(cleaned)
                if cls == "unknown":
                    continue
                patterns.append(BugPattern(
                    pattern_id=f"pennylane_{section_tag}_{md.stem}_{j}",
                    name=f"PennyLane {section_tag} in {md.stem}",
                    taxonomy_class=cls, description=cleaned[:800],
                    source="pennylane_changelog",
                    tags=["pennylane", "validated", section_tag, "pennylane"],
                ))
    return patterns

# IBM Runtime RST: extract Bug Fixes + Deprecation + Upgrade sections
def extract_ibm_runtime_entries(ibmr_root):
    patterns = []
    notes_dir = ibmr_root / "release-notes"
    if not notes_dir.exists():
        return patterns
    section_re = re.compile(
        r"(Bug Fixes|Deprecation Notes|Upgrade Notes)\s*\n[-=]+\s*\n(.*?)"
        r"(?=\n[A-Z][a-zA-Z\s]+\s*\n[-=]+|\Z)", re.DOTALL)
    for rst in notes_dir.glob("*.rst"):
        text = rst.read_text(encoding="utf-8", errors="replace")
        for sm in section_re.finditer(text):
            section_name, body = sm.group(1), sm.group(2)
            section_tag = "fixes" if "Bug" in section_name else "deprecations"
            for j, m in enumerate(re.finditer(r"\n-  +(.+?)(?=\n-  |\Z)", body, re.DOTALL)):
                item = m.group(1).strip()
                cleaned = re.sub(r"`([^`]+)`__", r"\1", item)
                cleaned = re.sub(r"\s+", " ", cleaned).strip()
                if _is_low_quality_entry(cleaned):
                    continue
                cls = classify_text_to_taxonomy(cleaned)
                if cls == "unknown":
                    continue
                patterns.append(BugPattern(
                    pattern_id=f"ibmr_{section_tag}_{rst.stem}_{j}",
                    name=f"IBM Runtime {section_tag} {rst.stem}",
                    taxonomy_class=cls, description=cleaned[:800],
                    source="ibm_runtime_changelog",
                    tags=["ibm_runtime", "validated", section_tag, "qiskit"],
                ))
    return patterns

# LintQ rules
LINTQ_RULES = [
    ("ql-unmeasurable-qubits", "measurement_error",
     "Qubits in a register never measured but circuit uses qasm_simulator. Unmeasured qubits cannot contribute to classical output."),
    ("ql-constant-classic-bit", "measurement_error",
     "Classical register bit read but never written by any measure always reads zero."),
    ("ql-operation-after-measurement", "measurement_error",
     "Quantum gate applied to a qubit AFTER measurement is meaningless on hardware."),
    ("ql-conditional-without-measurement", "measurement_error",
     "c_if referencing a cbit never written by measure: condition always reads zero."),
    ("ql-double-measurement", "measurement_error",
     "Same qubit measured twice with no intervening operation; second measure is redundant."),
    ("ql-measure-all-abuse", "measurement_error",
     "measure_all allocates new cbits, leading to mismatched cbit indexing afterward."),
    ("ql-oversized-circuit", "incorrect_operator",
     "QuantumCircuit initialised with more qubits than operations actually use."),
    ("ql-deprecated-identity", "incorrect_operator",
     "Code calls deprecated identity gate API like iden() that has been removed."),
    ("ql-ghost-composition", "incorrect_operator",
     "Subcircuit built but never composed into main circuit; usually incomplete construction."),
    ("ql-op-after-optimization", "incorrect_operator",
     "Gates added to circuit AFTER transpile bypass optimization and may break hardware constraints."),
]

def build_lintq_patterns():
    return [BugPattern(
        pattern_id=f"lintq_{rid}", name=rid, taxonomy_class=tax,
        description=desc, source="lintq_rules",
        tags=["lintq", "fse2024", "validated", "fixes", "qiskit"],
    ) for rid, tax, desc in LINTQ_RULES]

# Build full v6 KB
kb_patterns = []
kb_patterns += extract_yaml_releasenote_entries(WORK / "qiskit", "qiskit_releasenotes")
kb_patterns += extract_yaml_releasenote_entries(WORK / "qiskit_aer", "qiskit_aer_releasenotes")
kb_patterns += extract_yaml_releasenote_entries(WORK / "qiskit_ignis", "qiskit_ignis_releasenotes")
kb_patterns += extract_ibm_runtime_entries(WORK / "qiskit_ibm_runtime")
kb_patterns += extract_pennylane_changelog_entries(WORK / "pennylane")
kb_patterns += build_lintq_patterns()

src_dist = Counter(p.source for p in kb_patterns)
tax_dist = Counter(p.taxonomy_class for p in kb_patterns)
section_dist = Counter(t for p in kb_patterns for t in p.tags if t in {"fixes", "deprecations", "upgrade"})
fw_dist = Counter(t for p in kb_patterns for t in p.tags if t in {"qiskit", "pennylane", "cirq", "qsharp", "other"})

print(f"✓ v6 KB: {len(kb_patterns)} entries (was 1059 in v5)")
print(f"  By source:")
for s, n in src_dist.most_common():
    print(f"    {s:<28} {n}")
print(f"  By taxonomy:")
for c, n in tax_dist.most_common():
    print(f"    {c:<28} {n}")
print(f"  By section type:")
for c, n in section_dist.most_common():
    print(f"    {c:<28} {n}")
print(f"  By framework tag:")
for c, n in fw_dist.most_common():
    print(f"    {c:<28} {n}")


✓ v6 KB: 2040 entries (was 1059 in v5)
  By source:
    qiskit_releasenotes          1089
    pennylane_changelog          742
    qiskit_aer_releasenotes      152
    ibm_runtime_changelog        31
    qiskit_ignis_releasenotes    16
    lintq_rules                  10
  By taxonomy:
    incorrect_operator           1478
    measurement_error            279
    incorrect_qubit_mapping      194
    wrong_initial_state          76
    missing_barrier              13
  By section type:
    fixes                        1077
    deprecations                 579
    upgrade                      384
  By framework tag:
    pennylane                    1484
    qiskit                       1298


## Cell 7 — Framework-aware diversified retriever (v6)

**v6 change:** retrieval now considers the sample's framework. When a Bugs4Q sample is detected as Qiskit code, the retriever boosts Qiskit-tagged KB entries; when Cirq, it boosts Cirq/general entries. This stops the v5 problem where PennyLane-tagged patterns were retrieved for Qiskit code, adding noise.

Diversification still ensures multiple taxonomy classes are represented in the top-5.


In [7]:
import numpy as np
from rank_bm25 import BM25Okapi

def _tokenize(s):
    return [t for t in re.split(r"[^a-z0-9_]+", s.lower()) if t and len(t) > 1]

def detect_framework(code: str) -> str:
    """Identify which quantum framework a code snippet uses, via syntactic signatures."""
    code_l = code.lower()
    score = {"qiskit": 0, "cirq": 0, "pennylane": 0, "qsharp": 0}
    if "qiskit" in code_l: score["qiskit"] += 5
    if "quantumcircuit" in code_l: score["qiskit"] += 3
    if "qiskit_aer" in code_l or "qiskit-aer" in code_l: score["qiskit"] += 3
    if "aer.get_backend" in code_l or "aer_simulator" in code_l: score["qiskit"] += 2
    if "qasm" in code_l: score["qiskit"] += 1
    if re.search(r"\bcirq\.", code_l): score["cirq"] += 5
    if "gridqubit" in code_l or "lineqbit" in code_l: score["cirq"] += 3
    if "pennylane" in code_l or "qml." in code_l: score["pennylane"] += 5
    if "@qml.qnode" in code_l or "qml.device" in code_l: score["pennylane"] += 3
    # Q# detection: file extension hint or distinctive syntax
    if ".qs ===" in code_l[:300] or "operation " in code_l[:400] or "using (" in code_l[:400]:
        score["qsharp"] += 3
    best_fw, best_score = max(score.items(), key=lambda x: x[1])
    return best_fw if best_score >= 3 else "other"


class BugPatternRetriever:
    def __init__(self, patterns, top_pool=20):
        self.patterns = patterns
        self.top_pool = top_pool
        if patterns:
            corpus = [_tokenize(self._to_text(p)) for p in patterns]
            self.bm25 = BM25Okapi(corpus)
        else:
            self.bm25 = None
        # Pre-index by framework tag for fast lookup
        self._fw_index = defaultdict(list)
        for i, p in enumerate(patterns):
            for fw in {"qiskit", "pennylane", "cirq", "qsharp"} & set(p.tags):
                self._fw_index[fw].append(i)

    @staticmethod
    def _to_text(p):
        return " ".join(filter(None, [p.name, p.taxonomy_class, p.taxonomy_class,
                                       p.description, " ".join(p.tags)]))

    def retrieve(self, query, top_k=5, framework="other"):
        """v6: framework-aware retrieval with diversification.

        1. Compute BM25 scores over full KB
        2. If framework detected, BOOST scores for matching-framework entries by 1.5x
        3. Take top-12 from boosted ranking
        4. Diversify: pick at least one entry per distinct taxonomy class
        5. Fill remaining slots with next-best regardless of class
        """
        if self.bm25 is None:
            return []
        tokens = _tokenize(query)
        if not tokens:
            return []
        scores = self.bm25.get_scores(tokens)

        # v6: framework boost
        if framework in self._fw_index:
            boost_indices = set(self._fw_index[framework])
            scores = scores.copy()
            for i in boost_indices:
                scores[i] *= 1.5

        ranked = sorted(enumerate(scores), key=lambda x: -x[1])
        pool = [(i, s) for i, s in ranked[: self.top_pool] if s > 0]
        if not pool:
            return []
        result, seen = [], set()
        # First pass: best per distinct taxonomy class
        for i, _ in pool:
            p = self.patterns[i]
            if p.taxonomy_class not in seen:
                result.append(p)
                seen.add(p.taxonomy_class)
                if len(result) >= top_k:
                    break
        # Second pass: fill
        for i, _ in pool:
            if len(result) >= top_k:
                break
            if self.patterns[i] not in result:
                result.append(self.patterns[i])
        return result[:top_k]


retriever = BugPatternRetriever(kb_patterns, top_pool=20)

# Smoke test: same query for Qiskit code vs Cirq code should retrieve different patterns
test_qiskit = "qc = QuantumCircuit(2,2)\nqc.measure(0,0)\nqc.measure(0,0)"
test_cirq = "import cirq\nqubits = [cirq.GridQubit(0,0)]\ncirq.measure(qubits[0])"

print(f"✓ Retriever ready over {len(kb_patterns)} patterns with framework awareness")
print(f"\n  Test 1 — Qiskit code, framework={detect_framework(test_qiskit)}:")
for p in retriever.retrieve(test_qiskit, top_k=4, framework=detect_framework(test_qiskit)):
    fw_tag = next((t for t in p.tags if t in {"qiskit","pennylane","cirq","qsharp"}), "?")
    print(f"    [{p.taxonomy_class:<25}] [fw={fw_tag:<10}] {p.pattern_id[:55]}")
print(f"\n  Test 2 — Cirq code, framework={detect_framework(test_cirq)}:")
for p in retriever.retrieve(test_cirq, top_k=4, framework=detect_framework(test_cirq)):
    fw_tag = next((t for t in p.tags if t in {"qiskit","pennylane","cirq","qsharp"}), "?")
    print(f"    [{p.taxonomy_class:<25}] [fw={fw_tag:<10}] {p.pattern_id[:55]}")


✓ Retriever ready over 2040 patterns with framework awareness

  Test 1 — Qiskit code, framework=qiskit:
    [measurement_error        ] [fw=qiskit    ] qiskit_releasenotes_fixes_fix-optimize-swap-before-meas
    [wrong_initial_state      ] [fw=qiskit    ] qiskit_aer_releasenotes_fixes_fix-for-loop-no-parameter
    [incorrect_operator       ] [fw=qiskit    ] qiskit_releasenotes_fixes_fix-parameterized-self-invers
    [measurement_error        ] [fw=qiskit    ] qiskit_releasenotes_fixes_fix-reverse_bits-with-registe

  Test 2 — Cirq code, framework=cirq:
    [incorrect_operator       ] [fw=qiskit    ] qiskit_releasenotes_upgrade_dagcircuit-remove-deprecate
    [measurement_error        ] [fw=qiskit    ] lintq_ql-unmeasurable-qubits
    [incorrect_qubit_mapping  ] [fw=qiskit    ] qiskit_releasenotes_fixes_initial_layout_cp_none-8753d0
    [incorrect_operator       ] [fw=qiskit    ] qiskit_releasenotes_fixes_evogate-error-on-bad-list-b0b


## Cell 8 — Forced-choice classifier prompt (the v5 key fix)

**Critical changes:**
- Tells LLM the code DEFINITELY contains a bug (no abstaining)
- Asks for likelihood scores 0.0-1.0 for ALL 5 specific classes
- `taxonomy_class` is required to be the argmax of the scores
- `unknown` removed from output options entirely
- Server-side argmax fallback if model picks wrong class given its scores

**Both modes use the same prompt** — only the user message differs (rag includes retrieved patterns).

In [8]:
CLASSIFIER_PROMPT = """You are a quantum bug classifier. The given code DEFINITELY contains a bug. Your job is to identify which of these 5 categories the bug belongs to.

Categories (one of these MUST apply — there is no "no bug" option):
- incorrect_operator: wrong gate, wrong gate args (degrees vs radians, wrong matrix), deprecated/removed API, wrong API call, wrong type, wrong numerical computation, post-transpile mutation, oversized circuit, wrong sub-gate composition
- incorrect_qubit_mapping: wrong qubit/cbit index, off-by-one in register, MSB/LSB reversal, CNOT control/target swap, wrong identifier, wrong wires assignment
- missing_barrier: a barrier() is needed to prevent transpiler reordering but is absent
- wrong_initial_state: initialize() with wrong-size vector (must be 2^n), non-normalized amplitudes, wrong basis encoding, wrong state preparation
- measurement_error: missing measure before execute, double measure, gate-after-measure, c_if on unmeasured cbit, wrong qubit-to-cbit count

Score each category from 0.0 to 1.0 by how well it explains the bug. Scores DO NOT need to sum to 1.0. Pick the category with the HIGHEST score as the final classification.

Output JSON only, no markdown:
{"scores": {"incorrect_operator": <0-1>, "incorrect_qubit_mapping": <0-1>, "missing_barrier": <0-1>, "wrong_initial_state": <0-1>, "measurement_error": <0-1>}, "taxonomy_class": "<category with highest score>", "suspected_location": "<code fragment>", "justification": "<one sentence>"}"""

MAX_CODE_CHARS = 4000
MAX_REF_CHARS = 350

def _truncate(s, n):
    return s if len(s) <= n else s[:n] + "\n# [truncated]"

def build_prompt_only(sample):
    user = f"Code (definitely buggy):\n```python\n{_truncate(sample.code, MAX_CODE_CHARS)}\n```"
    return [{"role": "system", "content": CLASSIFIER_PROMPT},
            {"role": "user", "content": user}]

def build_rag_prompt(sample, retrieved):
    refs = []
    for i, p in enumerate(retrieved, 1):
        refs.append(f"Ref{i} [{p.taxonomy_class}]: {_truncate(p.description, MAX_REF_CHARS)}")
    refs_text = "\n".join(refs) if refs else "(no relevant references)"
    user = (
        f"Reference bug-fix patterns (from validated quantum-framework release notes — use these as category guidance):\n{refs_text}\n\n"
        f"Code (definitely buggy):\n```python\n{_truncate(sample.code, MAX_CODE_CHARS)}\n```"
    )
    return [{"role": "system", "content": CLASSIFIER_PROMPT},
            {"role": "user", "content": user}]

_demo = BugSample("demo", "test",
                  "qc = QuantumCircuit(2,2)\nqc.measure(0,0)\nqc.measure(0,0)\n",
                  "measurement_error")
_retrieved = retriever.retrieve(_demo.code, top_k=5)
print(f"✓ Forced-choice prompts ready")
print(f"  System: {len(CLASSIFIER_PROMPT)} chars")
print(f"  Demo prompt_only user: {len(build_prompt_only(_demo)[1]['content'])} chars")
print(f"  Demo rag user (with {len(_retrieved)} refs): {len(build_rag_prompt(_demo, _retrieved)[1]['content'])} chars")

✓ Forced-choice prompts ready
  System: 1482 chars
  Demo prompt_only user: 96 chars
  Demo rag user (with 5 refs): 2094 chars


## Cell 9 — LLM client (gpt-4o)

In [9]:
MODEL_NAME = "gpt-5.4"
TEMPERATURE = 0.0
MAX_TOKENS = 800  # bigger to fit scores dict

class BaseLLM:
    def complete(self, messages, **kw):
        raise NotImplementedError
    def parse(self, raw):
        try:
            t = raw.strip()
            if t.startswith("```"):
                lines = t.splitlines()
                t = "\n".join(lines[1:-1]) if len(lines) > 2 else t
            return json.loads(t)
        except json.JSONDecodeError as e:
            return {"_raw": raw[:500], "_parse_error": str(e)}

class MockLLM(BaseLLM):
    def __init__(self, seed=42): self.rng = random.Random(seed)
    def complete(self, messages, **kw):
        scores = {c: round(self.rng.uniform(0.1, 0.95), 2) for c in TAXONOMY_FORCED}
        return json.dumps({"scores": scores,
                          "taxonomy_class": max(scores, key=scores.get),
                          "suspected_location": "[Mock]",
                          "justification": "[Mock]"})

class OpenAILLM(BaseLLM):
    def __init__(self, api_key, model=MODEL_NAME):
        from openai import OpenAI
        self.client = OpenAI(api_key=api_key)
        self.model = model
    def complete(self, messages, **kw):
        from openai import RateLimitError, APIError
        for attempt in range(6):
            try:
                resp = self.client.chat.completions.create(
                    model=self.model, messages=messages,
                    temperature=kw.get("temperature", TEMPERATURE),
                    max_completion_tokens=kw.get("max_completion_tokens", MAX_TOKENS),
                )
                return resp.choices[0].message.content or ""
            except RateLimitError:
                w = min(2 ** attempt * 5, 120)
                print(f"  [openai] rate-limited, sleeping {w}s"); time.sleep(w)
            except APIError as e:
                if attempt == 5: raise
                time.sleep(min(2 ** attempt * 3, 60))
        raise RuntimeError("OpenAI retries exhausted")

USE_MOCK = False
if USE_MOCK:
    llm = MockLLM(seed=42); print("⚠ MockLLM")
else:
    try:
        from kaggle_secrets import UserSecretsClient
        api_key = UserSecretsClient().get_secret("OPENAI_API_KEY")
    except Exception:
        api_key = os.environ.get("OPENAI_API_KEY", "")
    if not api_key: raise RuntimeError("OPENAI_API_KEY missing")
    llm = OpenAILLM(api_key=api_key, model=MODEL_NAME)
    print(f"✓ OpenAI {MODEL_NAME} ready, T={TEMPERATURE}")
_resp = llm.complete([{"role":"system","content":"Reply exactly: OK"},
                      {"role":"user","content":"ping"}])
print(f"  smoke: {_resp[:60]!r}")

✓ OpenAI gpt-5.4 ready, T=0.0
  smoke: 'OK'


## Cell 10 — Evaluation orchestrator with framework-aware retrieval (v6)

The only change here from v5: `run_one_sample()` now detects the sample's framework and passes it to the retriever. Everything else (forced-choice argmax, top-2 tracking) is unchanged.


In [10]:
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score

RESULTS_DIR = WORK / "results_v6"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

def _argmax_class(scores_dict):
    if not isinstance(scores_dict, dict): return None
    valid = {k: v for k, v in scores_dict.items() if k in TAXONOMY_FORCED}
    if not valid: return None
    try:
        return max(valid.items(), key=lambda kv: float(kv[1]))[0]
    except (TypeError, ValueError):
        return None

def run_one_sample(sample, mode):
    retrieved = []
    if mode == "rag":
        # v6: detect framework first, pass to retriever
        framework = detect_framework(sample.code)
        retrieved = retriever.retrieve(sample.code, top_k=5, framework=framework)
        msgs = build_rag_prompt(sample, retrieved)
    else:
        msgs = build_prompt_only(sample)
    try:
        raw = llm.complete(msgs)
        parsed = llm.parse(raw)
    except Exception as exc:
        parsed = {"_parse_error": f"{type(exc).__name__}: {exc}"}

    scores_dict = parsed.get("scores", {})
    tax_from_argmax = _argmax_class(scores_dict)
    tax_from_llm = parsed.get("taxonomy_class")
    if tax_from_argmax:
        tax = tax_from_argmax
    elif tax_from_llm in TAXONOMY_FORCED:
        tax = tax_from_llm
    else:
        tax = "unknown"
    bug_likelihood = float(scores_dict.get(tax, 0.5)) if isinstance(scores_dict, dict) else 0.5
    bug_likelihood = max(0.0, min(1.0, bug_likelihood))

    diag = BugDiagnostic(
        sample_id=sample.sample_id, mode=mode,
        bug_likelihood=bug_likelihood, taxonomy_class=tax,
        class_scores={k: float(v) for k, v in scores_dict.items()
                      if k in TAXONOMY_FORCED and isinstance(v, (int, float))},
        suspected_location=str(parsed.get("suspected_location", ""))[:200],
        justification=str(parsed.get("justification", ""))[:600],
        ground_truth=sample.ground_truth,
        retrieved_patterns=[p.pattern_id for p in retrieved],
    )
    if diag.ground_truth is not None:
        diag.correct = (diag.taxonomy_class == diag.ground_truth)
    return diag

def compute_metrics(diags):
    labelled = [d for d in diags if d.ground_truth is not None]
    if not labelled: return {"n": 0}
    y_true = [d.ground_truth for d in labelled]
    y_pred = [d.taxonomy_class for d in labelled]
    classes = sorted(set(y_true))
    per_class = f1_score(y_true, y_pred, labels=classes, average=None, zero_division=0)
    top2_correct = 0
    for d in labelled:
        if d.class_scores:
            top2 = sorted(d.class_scores.items(), key=lambda kv: -kv[1])[:2]
            top2_classes = [c for c, _ in top2]
            if d.ground_truth in top2_classes:
                top2_correct += 1
    top2_acc = round(top2_correct / len(labelled), 4) if labelled else 0
    return {
        "n": len(labelled),
        "accuracy": round(accuracy_score(y_true, y_pred), 4),
        "top2_accuracy": top2_acc,
        "macro_f1": round(f1_score(y_true, y_pred, average="macro", zero_division=0), 4),
        "macro_p": round(precision_score(y_true, y_pred, average="macro", zero_division=0), 4),
        "macro_r": round(recall_score(y_true, y_pred, average="macro", zero_division=0), 4),
        "per_class_f1": {c: round(float(s), 4) for c, s in zip(classes, per_class)},
        "label_distribution": dict(Counter(y_true)),
        "prediction_distribution": dict(Counter(y_pred)),
    }

def evaluate(dataset_name, samples, mode):
    target = [s for s in samples if s.ground_truth]
    diags = []
    print(f"\n[{dataset_name} / {mode}] running on {len(target)} samples ...")
    for i, s in enumerate(target, 1):
        diag = run_one_sample(s, mode)
        diags.append(diag)
        if i % 10 == 0 or i == len(target):
            correct = sum(1 for d in diags if d.correct)
            print(f"  [{i:3d}/{len(target)}] accuracy: {correct}/{i} = {correct/i:.3f}")
    diag_path = RESULTS_DIR / f"diagnostics_{dataset_name}_{mode}.jsonl"
    with diag_path.open("w") as fh:
        for d in diags:
            fh.write(json.dumps(asdict(d), ensure_ascii=False) + "\n")
    metrics = compute_metrics(diags)
    (RESULTS_DIR / f"metrics_{dataset_name}_{mode}.json").write_text(json.dumps(metrics, indent=2))
    return diags, metrics

print("✓ v6 orchestrator ready (framework-aware retrieval + forced-choice + top-2)")


✓ v6 orchestrator ready (framework-aware retrieval + forced-choice + top-2)


## Cell 11 — Run evaluation

In [11]:
DATASETS = {"bugs4q": bugs4q_samples, "bugsqcp": bqcp_samples}
MODES = ["prompt_only", "rag"]

all_results = {}
for ds_name, samples in DATASETS.items():
    for mode in MODES:
        diags, metrics = evaluate(ds_name, samples, mode)
        all_results[(ds_name, mode)] = (diags, metrics)
print("\n✓ All evaluations complete.")


[bugs4q / prompt_only] running on 45 samples ...
  [ 10/45] accuracy: 6/10 = 0.600
  [ 20/45] accuracy: 14/20 = 0.700
  [ 30/45] accuracy: 17/30 = 0.567
  [ 40/45] accuracy: 25/40 = 0.625
  [ 45/45] accuracy: 28/45 = 0.622

[bugs4q / rag] running on 45 samples ...
  [ 10/45] accuracy: 3/10 = 0.300
  [ 20/45] accuracy: 11/20 = 0.550
  [ 30/45] accuracy: 15/30 = 0.500
  [ 40/45] accuracy: 24/40 = 0.600
  [ 45/45] accuracy: 29/45 = 0.644

[bugsqcp / prompt_only] running on 44 samples ...
  [ 10/44] accuracy: 8/10 = 0.800
  [ 20/44] accuracy: 16/20 = 0.800
  [ 30/44] accuracy: 23/30 = 0.767
  [ 40/44] accuracy: 30/40 = 0.750
  [ 44/44] accuracy: 34/44 = 0.773

[bugsqcp / rag] running on 44 samples ...
  [ 10/44] accuracy: 7/10 = 0.700
  [ 20/44] accuracy: 13/20 = 0.650
  [ 30/44] accuracy: 20/30 = 0.667
  [ 40/44] accuracy: 26/40 = 0.650
  [ 44/44] accuracy: 29/44 = 0.659

✓ All evaluations complete.


## Cell 12 — Summary (with top-2 accuracy)

In [12]:
import pandas as pd
rows = [{"dataset": ds, "mode": mode, "n": m["n"],
         "accuracy": m["accuracy"], "top2_acc": m["top2_accuracy"],
         "macro_f1": m["macro_f1"], "macro_p": m["macro_p"], "macro_r": m["macro_r"]}
        for (ds, mode), (_, m) in all_results.items()]
summary = pd.DataFrame(rows).sort_values(["dataset", "mode"]).reset_index(drop=True)
print("=" * 80 + "\nMAIN RESULTS (v5 — forced choice + cleaned KB)\n" + "=" * 80)
print(summary.to_string(index=False))

print("\n\nPER-CLASS F1\n" + "=" * 70)
for ds in DATASETS:
    print(f"\n[{ds}]")
    pc = {mode: all_results[(ds, mode)][1].get("per_class_f1", {}) for mode in MODES}
    print(pd.DataFrame(pc).fillna(0).round(3).to_string())

print("\n\nCLASS DISTRIBUTIONS (should be NO 'unknown' in predictions now)\n" + "=" * 70)
for ds in DATASETS:
    print(f"\n[{ds}]")
    for mode in MODES:
        m = all_results[(ds, mode)][1]
        print(f"  {mode:<12}")
        print(f"    true:      {dict(m.get('label_distribution', {}))}")
        print(f"    predicted: {dict(m.get('prediction_distribution', {}))}")

(RESULTS_DIR / "summary.json").write_text(json.dumps({
    "model": MODEL_NAME, "temperature": TEMPERATURE, "forced_choice": True,
    "kb_size": len(kb_patterns),
    "kb_sources": dict(Counter(p.source for p in kb_patterns)),
    "results": [{"dataset": ds, "mode": mode, **m} for (ds, mode), (_, m) in all_results.items()],
}, indent=2))
print("\n✓ summary.json written")

MAIN RESULTS (v5 — forced choice + cleaned KB)
dataset        mode  n  accuracy  top2_acc  macro_f1  macro_p  macro_r
 bugs4q prompt_only 45    0.6222    0.8000    0.3262   0.3333   0.3287
 bugs4q         rag 45    0.6444    0.8444    0.3193   0.3786   0.3148
bugsqcp prompt_only 44    0.7727    0.9091    0.4207   0.4947   0.3808
bugsqcp         rag 44    0.6591    0.8409    0.2282   0.2326   0.2242


PER-CLASS F1

[bugs4q]
                         prompt_only    rag
incorrect_operator             0.733  0.806
incorrect_qubit_mapping        0.000  0.000
measurement_error              0.571  0.471
wrong_initial_state            0.000  0.000

[bugsqcp]
                         prompt_only    rag
incorrect_operator             0.848  0.788
incorrect_qubit_mapping        0.588  0.353
missing_barrier                0.667  0.000


CLASS DISTRIBUTIONS (should be NO 'unknown' in predictions now)

[bugs4q]
  prompt_only 
    true:      {'measurement_error': 12, 'wrong_initial_state': 5, 'incorre

## Cell 13 — Paired comparison

In [13]:
for ds in DATASETS:
    po = {d.sample_id: d for d in all_results[(ds, "prompt_only")][0]}
    rg = {d.sample_id: d for d in all_results[(ds, "rag")][0]}
    common = set(po) & set(rg)
    table = {"both_correct": 0, "both_wrong": 0, "rag_only": 0, "po_only": 0}
    for sid in common:
        p, r = po[sid].correct, rg[sid].correct
        if p is None or r is None: continue
        if p and r: table["both_correct"] += 1
        elif not p and not r: table["both_wrong"] += 1
        elif r and not p: table["rag_only"] += 1
        elif p and not r: table["po_only"] += 1
    n = sum(table.values())
    print(f"\n[{ds}] paired over {n}:")
    for k, v in table.items():
        print(f"  {k:<18} {v:>3}" + (f"  ({100*v/n:.1f}%)" if n else ""))
    print(f"  net RAG advantage: {table['rag_only'] - table['po_only']:+d}")


[bugs4q] paired over 45:
  both_correct        25  (55.6%)
  both_wrong          13  (28.9%)
  rag_only             4  (8.9%)
  po_only              3  (6.7%)
  net RAG advantage: +1

[bugsqcp] paired over 44:
  both_correct        29  (65.9%)
  both_wrong          10  (22.7%)
  rag_only             0  (0.0%)
  po_only              5  (11.4%)
  net RAG advantage: -5


## Cell 14 — Disagreement inspection (with class scores)

In [14]:
for ds in DATASETS:
    po = {d.sample_id: d for d in all_results[(ds, "prompt_only")][0]}
    rg = {d.sample_id: d for d in all_results[(ds, "rag")][0]}
    disagreements = [sid for sid in (set(po) & set(rg))
                     if po[sid].taxonomy_class != rg[sid].taxonomy_class]
    print(f"\n[{ds}] {len(disagreements)} disagreements")
    for sid in disagreements[:3]:
        p, r = po[sid], rg[sid]
        print(f"\n  {sid}  (gt={p.ground_truth})")
        print(f"    po:  {p.taxonomy_class:<25} ({'✓' if p.correct else '✗'})  scores={ {k: round(v,2) for k,v in p.class_scores.items()} }")
        print(f"    rag: {r.taxonomy_class:<25} ({'✓' if r.correct else '✗'})  scores={ {k: round(v,2) for k,v in r.class_scores.items()} }")
        print(f"    rag retrieved: {r.retrieved_patterns[:3]}")
        print(f"    rag says: {r.justification[:150]}")


[bugs4q] 9 disagreements

  bugs4q_0004  (gt=incorrect_operator)
    po:  incorrect_operator        (✓)  scores={'incorrect_operator': 0.92, 'incorrect_qubit_mapping': 0.18, 'missing_barrier': 0.05, 'wrong_initial_state': 0.0, 'measurement_error': 0.0}
    rag: incorrect_qubit_mapping   (✗)  scores={'incorrect_operator': 0.18, 'incorrect_qubit_mapping': 0.97, 'missing_barrier': 0.03, 'wrong_initial_state': 0.0, 'measurement_error': 0.0}
    rag retrieved: ['qiskit_releasenotes_fixes_pauli_evolve_rz-f6d9fc0cabd8a183_0', 'qiskit_releasenotes_upgrade_rework-circuit-argument-resolver-780091cd6f97f872_0', 'qiskit_releasenotes_fixes_fix-initial_layout-loose-qubits-0c59b2d6fb99d7e6_0']
    rag says: The bug is caused by appending/inserting a circuit built on different qubits into another circuit, which is fundamentally a wrong qubit/wire assignmen

  bugs4q_0043  (gt=wrong_initial_state)
    po:  incorrect_operator        (✗)  scores={'incorrect_operator': 0.92, 'incorrect_qubit_mapping': 0.

## Cell 15 — Package results

In [15]:
import shutil
archive = shutil.make_archive(str(WORK / "rag_evaluation_results_v5"), "zip", root_dir=RESULTS_DIR)
print(f"✓ {archive} ({Path(archive).stat().st_size/1024:.1f} KB)")
for p in sorted(RESULTS_DIR.iterdir()):
    print(f"  {p.name}  ({p.stat().st_size} bytes)")

✓ /kaggle/working/rag_evaluation_results_v5.zip (31.6 KB)
  diagnostics_bugs4q_prompt_only.jsonl  (30079 bytes)
  diagnostics_bugs4q_rag.jsonl  (47328 bytes)
  diagnostics_bugsqcp_prompt_only.jsonl  (31055 bytes)
  diagnostics_bugsqcp_rag.jsonl  (46500 bytes)
  metrics_bugs4q_prompt_only.json  (592 bytes)
  metrics_bugs4q_rag.json  (565 bytes)
  metrics_bugsqcp_prompt_only.json  (557 bytes)
  metrics_bugsqcp_rag.json  (528 bytes)
  summary.json  (3185 bytes)
